# Debugging code
## Week 4: Discussion section

1. Load the data and agree on an expectation

In [11]:
import io
import pandas as pd

CSV = """site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
SB-01,2024-01-15,12.4,"1,240",0.82,good
SB-01,2024-02-15,13.1,"1,180",0.79,good
SB-01,2024-03-15,15.8,"2,050",1.21,good
SB-01,2024-04-15,18.2,"1,620",0.94,good
sb-01,2024-05-15,21.5,"980",0.66,good
SB-01 ,2024-06-15,-999,"540",0.51,suspect
SB-02,2024-01-15,11.8,"3,410",1.88,good
SB-02,2024-02-15,12.3,"3,290",1.92,good
SB-02,2024-03-15,14.9,"4,870",2.41,good
SB-02,2024-04-15,17.6,"3,950",2.09,estimated
SB-02,2024-05-15,20.9,"2,240",1.44,good
sb-02,2024-06-15,-999,"1,130",1.02,suspect
SB-03,2024-01-15,10.2,"620",<0.05,good
SB-03,2024-02-15,10.9,"590",<0.05,good
SB-03,2024-03-15,13.4,"1,040",0.33,good
SB-03,2024-04-15,16.1,"870",0.28,good
SB-03 ,2024-05-15,19.3,"410",<0.05,good
SB-03,2024-06-15,-999,"180",<0.05,suspect
"""

stream = pd.read_csv(io.StringIO(CSV))
stream.head(18)

,site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
0,SB-01,2024-01-15,12.4,"1,240",0.82,good
1,SB-01,2024-02-15,13.1,"1,180",0.79,good
2,SB-01,2024-03-15,15.8,"2,050",1.21,good
3,SB-01,2024-04-15,18.2,"1,620",0.94,good
4,sb-01,2024-05-15,21.5,980,0.66,good
5,SB-01,2024-06-15,-999.0,540,0.51,suspect
6,SB-02,2024-01-15,11.8,"3,410",1.88,good
7,SB-02,2024-02-15,12.3,"3,290",1.92,good
8,SB-02,2024-03-15,14.9,"4,870",2.41,good
9,SB-02,2024-04-15,17.6,"3,950",2.09,estimated


Before writing any more code, agree with your team on the answers to these three questions and write them in a markdown cell.

- How many monitoring sites are in this dataset?
    - Three monitoring sites
        - one site ID in lower case rather than upper case
- What is a plausible range for stream water temperature in degrees Celsius?
    - ~ 10-21.5 degC
- Roughly what mean water temperature do you expect across all measurements?
    - 15 degC

In [12]:
stream.dtypes

site_id           object
date              object
water_temp_c     float64
discharge_cfs     object
nitrate_mg_l      object
qc_flag           object
dtype: object

In [13]:
stream['discharge_cfs'] * 0.0283168

TypeError: can't multiply sequence by non-int of type 'float'

In [ ]:
stream['discharge_cfs'].unique()

array(['1,240', '1,180', '2,050', '1,620', '980', '540', '3,410', '3,290',
       '4,870', '3,950', '2,240', '1,130', '620', '590', '1,040', '870',
       '410', '180'], dtype=object)

In [ ]:
discharge_cms = (stream['discharge_cfs']
                    .str.replace(',', '', regex=False)
                    .astype(float)) * 0.0283168

discharge_cms.head(3)

0    35.112832
1    33.413824
2    58.049440
Name: discharge_cfs, dtype: float64

In [15]:
assert pd.api.types.is_numeric_dtype(discharge_cms), 'Discharge failed to convert to numeric'

In [ ]:
# Reprex of problem
# import pandas as pd
# df = pd.DataFrame({'discharge_cfs': ['1,240', '980']})
# df['discharge_cfs'] * 0.0283168     # TypeError: can't multiply sequence by non-int

TypeError: can't multiply sequence by non-int of type 'float'


| What I expected |	What happened |	My hypothesis |	What I changed | Result |
| :---            | :----:        | :----:        | :----:         | ---:   |
| Discharge in m³/s, around 35	| TypeError: can't multiply sequence by non-int | 	discharge_cfs is text because of thousands separators |	.str.replace(',', '') then .astype(float) |	Works, 35.11 m³/s, guard added |

In [14]:
stream['site_id'].strip()

AttributeError: 'Series' object has no attribute 'strip'

In [16]:
stream['water_temp_c'].mean()

-153.8111111111111

The output is different than expected. The value -153 would be an absurdly cold temperature. This is likely due to the -999 values in the dataframe.

In [ ]:
stream['water_temp_c'].unique() 
# Find -999 value

array([  12.4,   13.1,   15.8,   18.2,   21.5, -999. ,   11.8,   12.3,
         14.9,   17.6,   20.9,   10.2,   10.9,   13.4,   16.1,   19.3])

In [18]:
# Assigning NAs at the moment we read in the CSV
stream = pd.read_csv(io.StringIO(CSV), na_values = {'water_temp_c' : [-999]})

In [19]:
stream['water_temp_c'].mean()

15.226666666666668